# Notebook 04 — CRSP Data Pull

First live pull from WRDS/CRSP. Verifies the `src.data.crsp` module works against the real database.

**Run once to confirm access, then save the output panel to `data/processed/` for all subsequent work.**

You will be prompted for WRDS credentials on first run. After entering them you'll be offered the option to save to `.pgpass` — say yes, so you're never prompted again.

In [1]:
import sys
sys.path.append('..')  # so 'from src...' resolves from the notebooks/ folder

import pandas as pd
import wrds
from src.data.crsp import load_crsp_daily

## Step 1 — connect and verify access

First call opens the connection and prompts for credentials if .pgpass isn't set up yet.

In [2]:
db = wrds.Connection()

# Confirm the CRSP schema is accessible
tables = db.list_tables(library='crsp')
print('CRSP tables available:', tables[:10], '...')

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
CRSP tables available: ['acti', 'asia', 'asib', 'asic', 'asio', 'asix', 'bmdebt', 'bmheader', 'bmpaymts', 'bmquotes'] ...


## Step 2 — small test pull (1 year) to confirm the query works

Don't pull 2010–2024 in one go until you've confirmed a small pull works correctly.
This fetches one year — should return ~125k rows (≈500 stocks × 252 trading days).

In [3]:
test_panel = load_crsp_daily(start='2023-01-01', end='2023-12-31', db=db)

print('Shape:', test_panel.shape)
print('Columns:', test_panel.columns.tolist())
print()
print(test_panel.head(10))

Shape: (114663, 5)
Columns: ['permno', 'date', 'ticker', 'ret', 'mktcap']

   permno       date ticker       ret        mktcap
0   10104 2023-01-03   ORCL  0.024223  225730301.16
1   10104 2023-01-04   ORCL  0.009078  227779453.44
2   10104 2023-01-05   ORCL -0.002012  227321090.43
3   10104 2023-01-06   ORCL  0.016012  230961031.98
4   10104 2023-01-09   ORCL  0.012608  233010184.26
5   10104 2023-01-10   ORCL  0.000926   233225884.5
6   10104 2023-01-11   ORCL  0.024277   238888015.8
7   10104 2023-01-12   ORCL  0.002032  239373341.34
8   10104 2023-01-13   ORCL  0.004731   240505767.6
9   10104 2023-01-17   ORCL -0.006726   238888015.8


In [4]:
# Sanity checks on the test pull
print('--- Sanity checks ---')
print('Unique permnos:', test_panel['permno'].nunique(), '(expect ~400-500)')
print('Date range:', test_panel['date'].min(), 'to', test_panel['date'].max())
print('Missing ret:', test_panel['ret'].isna().sum(), 'rows')
print('Negative prc rows (should be 0 — prc column dropped):', 'prc' in test_panel.columns)
print()

# Confirm return scale: daily returns should mostly be in (-10%, +10%)
ret = test_panel['ret'].dropna()
print('Return range:', ret.min().round(4), 'to', ret.max().round(4))
print('% of returns outside ±10%:', (ret.abs() > 0.10).mean().round(4), '(should be small)')

--- Sanity checks ---
Unique permnos: 464 (expect ~400-500)
Date range: 2023-01-03 00:00:00 to 2023-12-29 00:00:00
Missing ret: 6 rows
Negative prc rows (should be 0 — prc column dropped): False

Return range: -0.9048 to 0.2947
% of returns outside ±10%: 0.0025 (should be small)


## Step 3 — check delisting returns were merged in

If the delisting merge worked, some rows will have large negative returns (delistings due to bankruptcy, etc.).
This is correct — it's the survivorship-bias fix.

In [5]:
# Large negative returns are expected and desired — they represent delistings
large_neg = test_panel[test_panel['ret'] < -0.50]
print('Rows with ret < -50% (likely delistings):', len(large_neg))
print(large_neg[['permno', 'ticker', 'date', 'ret']].to_string())

Rows with ret < -50% (likely delistings): 4
      permno ticker       date       ret
3295   11786   SIVB 2023-03-09 -0.604077
3296   11786   SIVB 2023-03-10 -0.628725
5344   12448    FRC 2023-03-13 -0.618273
5377   12448    FRC 2023-04-28 -0.904843


## Step 4 — full pull (2010–2024)

Only run this once the test pull above looks correct.
This is the full proposal dataset — may take several minutes depending on connection speed.

**Warning:** do not commit the saved parquet file to git. It's in `data/processed/` which is already in `.gitignore`.

In [6]:
# Uncomment and run once the test pull above is confirmed correct

panel = load_crsp_daily(start='2010-01-01', end='2024-12-31', db=db)
print('Full panel shape:', panel.shape)
print('Unique permnos:', panel['permno'].nunique())
print('Date range:', panel['date'].min(), 'to', panel['date'].max())

# Save to parquet — fast read/write, preserves dtypes, far smaller than CSV
panel.to_parquet('../data/processed/crsp_daily_2010_2024.parquet', index=False)
print('Saved.')

Full panel shape: (2235322, 5)
Unique permnos: 744
Date range: 2010-01-04 00:00:00 to 2024-12-31 00:00:00
Saved.


In [7]:
# Always close the connection when done
db.close()